# Project 04 — SaaS Churn & Revenue Leakage Analysis

**Dataset:** IBM Telco Customer Churn — [Download from Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

Save as `data/telco_churn.csv`. The notebook also generates synthetic data if the file is missing.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, roc_curve
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.facecolor'] = '#111'
matplotlib.rcParams['figure.facecolor'] = '#0a0a0a'
matplotlib.rcParams['text.color'] = '#f0ede8'
matplotlib.rcParams['axes.labelcolor'] = '#a09d98'
matplotlib.rcParams['xtick.color'] = '#5a5755'
matplotlib.rcParams['ytick.color'] = '#5a5755'
matplotlib.rcParams['axes.edgecolor'] = '#2a2a2a'
matplotlib.rcParams['grid.color'] = '#1e1e1e'
print('Libraries loaded.')

## Step 1 — Load Data

In [ ]:
try:
    df = pd.read_csv('data/telco_churn.csv')
    df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
    print(f'Real Telco data loaded: {len(df)} customers, churn rate: {df["Churn"].mean()*100:.1f}%')
except FileNotFoundError:
    print('Telco data not found — generating synthetic data...')
    np.random.seed(42)
    n = 7043
    contract = np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55, 0.24, 0.21])
    tenure = np.clip(np.random.exponential(25, n), 1, 72).astype(int)
    monthly_charges = np.random.normal(65, 30, n).clip(20, 120)
    has_tech_support = np.random.choice([0, 1], n, p=[0.5, 0.5])
    internet_service = np.random.choice(['Fiber optic','DSL','No'], n, p=[0.44, 0.34, 0.22])
    # Churn probability model
    churn_logit = (
        + (contract == 'Month-to-month') * 1.8
        - tenure * 0.04
        + monthly_charges * 0.015
        - has_tech_support * 0.6
        + (internet_service == 'Fiber optic') * 0.4
        + np.random.normal(0, 0.5, n)
    )
    churn_prob = 1 / (1 + np.exp(-churn_logit))
    churn = (np.random.uniform(0, 1, n) < churn_prob).astype(int)
    df = pd.DataFrame({
        'customerID': [f'CUST-{i:05d}' for i in range(n)],
        'Contract': contract,
        'tenure': tenure,
        'MonthlyCharges': monthly_charges.round(2),
        'TotalCharges': (monthly_charges * tenure).round(2),
        'TechSupport': np.where(has_tech_support, 'Yes', 'No'),
        'InternetService': internet_service,
        'gender': np.random.choice(['Male','Female'], n),
        'SeniorCitizen': np.random.choice([0,1], n, p=[0.84,0.16]),
        'Partner': np.random.choice(['Yes','No'], n),
        'Dependents': np.random.choice(['Yes','No'], n, p=[0.3, 0.7]),
        'PhoneService': np.random.choice(['Yes','No'], n, p=[0.9,0.1]),
        'PaperlessBilling': np.random.choice(['Yes','No'], n, p=[0.59,0.41]),
        'PaymentMethod': np.random.choice(['Electronic check','Mailed check','Bank transfer','Credit card'], n),
        'Churn': churn
    })
    print(f'Synthetic data: {len(df)} customers, churn rate: {df["Churn"].mean()*100:.1f}%')

## Step 2 — SQL-Style Cohort Analysis

In [ ]:
# Tenure bands
df['tenure_band'] = pd.cut(df['tenure'], bins=[0, 12, 24, 72], labels=['0-12 mo','12-24 mo','24+ mo'])

cohort = df.groupby(['Contract','tenure_band'])['Churn'].agg(['mean','count']).reset_index()
cohort.columns = ['Contract','Tenure Band','Churn Rate','Count']
cohort['Churn Rate (%)'] = (cohort['Churn Rate'] * 100).round(1)

print('Churn Rate by Contract Type and Tenure Band:')
print(cohort[['Contract','Tenure Band','Churn Rate (%)','Count']].sort_values('Churn Rate (%)', ascending=False).to_string(index=False))

# Pivot for heatmap
pivot = cohort.pivot(index='Contract', columns='Tenure Band', values='Churn Rate (%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(pivot.values, cmap='RdYlGn_r', aspect='auto')
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels(pivot.columns)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.iloc[i, j]
        if not pd.isna(val):
            axes[0].text(j, i, f'{val:.1f}%', ha='center', va='center', color='#0a0a0a', fontweight='bold', fontsize=11)
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Churn Rate Heatmap by Segment', color='#f0ede8', fontsize=12)

contract_churn = df.groupby('Contract')['Churn'].mean() * 100
axes[1].bar(contract_churn.index, contract_churn.values, color=['#f07060','#f0b860','#c8f060'], alpha=0.85, edgecolor='#0a0a0a')
axes[1].set_title('Churn Rate by Contract Type', color='#f0ede8', fontsize=12)
axes[1].set_ylabel('Churn Rate (%)')
plt.tight_layout()
plt.savefig('churn_cohort_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Classification Model (Logistic Regression + Random Forest)

In [ ]:
# Feature engineering
df_model = df.copy()
cat_cols = ['Contract','TechSupport','InternetService','gender','Partner','Dependents','PhoneService','PaperlessBilling','PaymentMethod']
le = LabelEncoder()
for col in cat_cols:
    if col in df_model.columns:
        df_model[col] = le.fit_transform(df_model[col].astype(str))

feature_cols = ['Contract','tenure','MonthlyCharges','TotalCharges','TechSupport','InternetService','SeniorCitizen']
feature_cols = [c for c in feature_cols if c in df_model.columns]

X = df_model[feature_cols]
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=500)
lr.fit(X_train_s, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test_s))
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_s)[:,1])

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])

print(f'Logistic Regression — Accuracy: {lr_acc:.3f} | AUC: {lr_auc:.3f}')
print(f'Random Forest       — Accuracy: {rf_acc:.3f} | AUC: {rf_auc:.3f}')

# ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr.predict_proba(X_test_s)[:,1])
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf.predict_proba(X_test)[:,1])

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Reg (AUC={lr_auc:.3f})', color='#60a8f0', linewidth=2)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})', color='#c8f060', linewidth=2)
ax.plot([0,1],[0,1], '--', color='#5a5755', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Churn Prediction Models', color='#f0ede8', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — SHAP Explainability & Revenue-at-Risk Ranking

In [ ]:
# Feature importance (built-in RF — always works)
fi = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
print('Feature Importance (Random Forest):')
print(fi.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi['Feature'][::-1], fi['Importance'][::-1], color='#c8f060', alpha=0.85, edgecolor='#0a0a0a')
ax.set_title('Feature Importance — Churn Drivers (Random Forest)', color='#f0ede8', fontsize=12)
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Try SHAP
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap_means = np.abs(sv).mean(axis=0)
    shap_df = pd.DataFrame({'Feature': feature_cols, 'Mean |SHAP|': shap_means}).sort_values('Mean |SHAP|', ascending=False)
    print('\nSHAP Feature Importance:')
    print(shap_df.to_string(index=False))
    shap.summary_plot(sv, X_test, feature_names=feature_cols, plot_type='bar', show=False)
    plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('\nSHAP not installed. Run: pip install shap')
    print('Using Random Forest feature importance as equivalent for portfolio.')

In [ ]:
# Revenue-at-Risk ranking
churn_probs = rf.predict_proba(X_test)[:,1]
test_df = X_test.copy()
test_df['churn_prob'] = churn_probs
test_df['annual_arr'] = (df_model.loc[X_test.index, 'MonthlyCharges'] * 12).values
test_df['revenue_at_risk'] = test_df['churn_prob'] * test_df['annual_arr']
test_df = test_df.sort_values('revenue_at_risk', ascending=False)

top60_risk = test_df.head(60)['revenue_at_risk'].sum()
print(f'\nTop 60 accounts by Revenue-at-Risk: ${top60_risk:,.0f}')
print(f'Average churn probability in top 60: {test_df.head(60)["churn_prob"].mean():.1%}')

# Export for stakeholder review
test_df.head(100)[['churn_prob','annual_arr','revenue_at_risk']].to_csv('revenue_at_risk_accounts.csv', index=True)
print('\nTop 100 at-risk accounts exported to revenue_at_risk_accounts.csv')
print('\n=== PROJECT 04 COMPLETE ===')
print(f'Best model: Random Forest | Accuracy: {rf_acc:.1%} | AUC: {rf_auc:.3f}')
print(f'Top 60 accounts: ${top60_risk:,.0f} ARR at risk')